# Mudik 1448 H: bentuk puncaknya

Musim mudik 1448 H sedang berjalan saat edisi ini terbit. Sebelum ikut padat, layak ditanya lebih dulu: seperti apa sebenarnya bentuk puncak musim ini? Dua musim Idulfitri terakhir yang tercatat rapi (1445 H dan 1446 H) menyimpan jawabannya dalam satu tabel harian yang sama, dan bentuknya ternyata bukan bukit tunggal.

In [ ]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd

from tools import gaya

gaya.terapkan()

df = pd.read_csv("data/pergerakan-event.csv", parse_dates=["tanggal"])
print(df["event"].value_counts().to_dict())
print("moda:", sorted(df["moda"].unique()))

EV_ID = ["ANGLEB 2024", "ANGLEB 2025", "NATARU 2024 2025"]
LAB = {"ANGLEB 2024": "Idulfitri 1445 H (2024)",
       "ANGLEB 2025": "Idulfitri 1446 H (2025)",
       "NATARU 2024 2025": "Nataru 2024/25"}
harian = df.groupby(["event", "hari_relatif"])["penumpang"].sum().unstack(0)
print((harian / 1e3).round(0))

{'ANGLEB 2024': 110, 'ANGLEB 2025': 110, 'NATARU 2024 2025': 105}
moda: ['Angkutan Jalan', 'Angkutan Laut', 'Angkutan Perkeretaapian', 'Angkutan SDP', 'Angkutan Udara']
event         ANGLEB 2024  ANGLEB 2025  NATARU 2024 2025
hari_relatif                                            
-10                 688.0        791.0             718.0
-9                  596.0        864.0             642.0
-8                  605.0        925.0             623.0
-7                  736.0        902.0             787.0
-6                  857.0        948.0             865.0
-5                 1070.0       1054.0             930.0
-4                 1231.0       1143.0             987.0
-3                 1249.0       1222.0            1012.0
-2                 1162.0       1057.0             980.0
-1                  916.0        900.0             949.0
 0                  682.0        750.0             916.0
 1                  869.0        981.0               NaN
 2                  958.0       1

## Data & cara ukur

Sumbernya satu katalog terbuka Kemenhub (HUBNET): penumpang harian lima moda angkutan umum (jalan, kereta api, udara, laut, dan sungai-danau-penyeberangan) pada jendela pantauan posko H-10 sampai H+10/11. Angkutan umum memang bukan seluruh mudik (mobil-motor pribadi tidak terhitung), tapi ia mencatat irama musim dengan paling disiplin: tabel yang sama, tiga musim berurutan. Katalog memakai penanda hari Lebaran ganda (`H 1`, `H 2`), yang saya petakan jadi `hari_relatif`: H 1 ke 0, H 2 ke 1, `H + k` ke 1+k, `H - k` ke -k. Metriknya: bentuk harian (bagian tiap hari dari total musim), hari puncak tiap moda, bagian pergerakan di sisi sesudah Lebaran, dan hari median musim (hari ketika separuh total pergerakan terlewati). Katalog disimpan mentah dan diambil ulang lewat `tools/data_scraping.py`.

Angka pembanding musim 1447 H (Maret 2026) belum masuk katalog mesin-baca;
ia hanya hadir sebagai realisasi rilis posko di `data/README.md`.

In [2]:
# dasar verifikasi: setiap musim penuh 21-22 hari, tanpa tanggal ganda
for ev in EV_ID:
    g = df[df.event == ev]
    assert not g.duplicated(subset=["moda", "hari_relatif"]).any(), ev
    print(ev, "|", g["hari_relatif"].nunique(), "hari |",
          f"total {g.penumpang.sum():,.0f} penumpang-hari")

def profil(ev):
    s = harian[ev].dropna()
    return {"total": s.sum(), "puncak_hari": int(s.idxmax()), "puncak": s.max(),
            "mudik_min_sebelum": s.loc[-9], "hari_H": s.loc[0],
            "sesudah_h2": s.loc[2:].sum(), "sesudah_share": s.loc[2:].sum() / s.sum()}

for ev in EV_ID:
    print(ev, {k: (round(v, 3) if isinstance(v, float) and v < 1 else v) for k, v in profil(ev).items()})

ANGLEB 2024 | 22 hari | total 21,468,508 penumpang-hari
ANGLEB 2025 | 22 hari | total 23,062,930 penumpang-hari
NATARU 2024 2025 | 21 hari | total 18,319,399 penumpang-hari
ANGLEB 2024 {'total': np.float64(21468508.0), 'puncak_hari': 4, 'puncak': np.float64(1350087.0), 'mudik_min_sebelum': np.float64(596387.0), 'hari_H': np.float64(681775.0), 'sesudah_h2': np.float64(10808599.0), 'sesudah_share': np.float64(0.503)}
ANGLEB 2025 {'total': np.float64(23062930.0), 'puncak_hari': 6, 'puncak': np.float64(1444983.0), 'mudik_min_sebelum': np.float64(864352.0), 'hari_H': np.float64(750203.0), 'sesudah_h2': np.float64(11524775.0), 'sesudah_share': np.float64(0.5)}
NATARU 2024 2025 {'total': np.float64(18319399.0), 'puncak_hari': 5, 'puncak': np.float64(1014842.0), 'mudik_min_sebelum': np.float64(642178.0), 'hari_H': np.float64(915565.0), 'sesudah_h2': np.float64(8910569.0), 'sesudah_share': np.float64(0.486)}


## Pembedahan

Gambar pertama memperlihatkan bentuk dua musim Idulfitri itu dalam satu bidang.

In [3]:
def label_h(h):
    if h <= -1:
        return f"H{h}"
    if h == 0:
        return "H"
    if h == 1:
        return "H2"
    return f"H+{h - 1}"

xs = harian.index.values

def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Puncak mudik itu dua bukit, dan bukit keduanya lebih tinggi",
        "Penumpang harian angkutan umum dua musim Idulfitri beruntun. Bukit sesudah "
        "Lebaran selalu mengalahkan bukit mudik, di kedua musim.",
        "HUBNET kemenhub, akses 30 Sep 2026 (angkutan umum; kendaraan pribadi di luar tabel)",
        220,
    )
    ax.plot(xs, harian["ANGLEB 2024"] / 1e3, color=gaya.INK2, marker="o", ms=4,
            lw=1.6, label="1445 H (2024)")
    ax.plot(xs, harian["ANGLEB 2025"] / 1e3, color=gaya.AKSEN, marker="o", ms=4,
            lw=2.0, label="1446 H (2025)")
    ax.axvspan(-1.6, 1.6, color=gaya.KERTAS2 if hasattr(gaya, "KERTAS2") else "#e8e4da",
               alpha=0.55, zorder=0)
    ax.text(0.0, 1485, "hari H\n(lembah)", ha="center", fontsize=8.6 * fs,
            color=gaya.INK2)
    ax.annotate("bukit mudik\nH-3: 1,22-1,25 jt", xy=(-3, 1245), xytext=(-10.2, 1340),
                fontsize=8.6 * fs, color=gaya.INK, ha="left",
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9))
    ax.annotate("bukit balik\n1445 H: H+4 = 1,35 jt\n1446 H: H+6 = 1,44 jt",
                xy=(5.5, 1440), xytext=(5.8, 1545), fontsize=8.6 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9), ha="left")
    ax.set_ylabel("penumpang angkutan umum (ribu per hari)", fontsize=9.5 * fs)
    ax.set_xticks(xs)
    ax.set_xticklabels([label_h(h) for h in xs], fontsize=8 * fs)
    ax.set_ylim(0, 1750)
    ax.legend(fontsize=8.6 * fs, loc="lower center", frameon=False, ncol=2)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    gaya.simpan(fig, "01-bentuk-angleb", mode)


for mode in gaya.MODE:
    grafik_1(mode)

Pertanyaan lanjut: moda mana yang membentuk kedua bukit itu, dan bagaimana runduknya musim 1446 H (2025) yang kalender liburnya diperpanjang cuti bersama? Gambar kedua menandai hari paling padat tiap moda.

In [4]:
puncak_moda = (df[df.event.str.startswith("ANGLEB")]
               .sort_values("penumpang")
               .groupby(["event", "moda"])
               .tail(1)
               .sort_values(["moda", "event"]))
print(puncak_moda[["event", "moda", "hari_relatif", "penumpang"]].to_string(index=False))

MODA = ["Angkutan Jalan", "Angkutan Perkeretaapian", "Angkutan Udara", "Angkutan Laut", "Angkutan SDP"]

def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Kapan puncak tiap moda: di 2025 semua bergeser ke sesudah Lebaran",
        "Hari paling padat tiap moda. 2024: dua moda memuncak sebelum hari H. "
        "2025: hampir semua menumpuk H+6, hanya laut menahan di H+3.",
        "HUBNET kemenhub; panah menghubungkan moda yang sama dari 2024 ke 2025",
        10,
    )
    yk = sorted(MODA)
    ys = {m: i for i, m in enumerate(yk)}
    p24 = puncak_moda[puncak_moda.event == "ANGLEB 2024"].set_index("moda")
    p25 = puncak_moda[puncak_moda.event == "ANGLEB 2025"].set_index("moda")
    for m in yk:
        a, b = p24.loc[m, "hari_relatif"], p25.loc[m, "hari_relatif"]
        ax.annotate("", xy=(b, ys[m] + 0.16), xytext=(a, ys[m] + 0.16),
                    arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9, alpha=0.7))
    ax.scatter(p24["hari_relatif"].reindex(yk), [ys[m] - 0.16 for m in yk],
               s=90, facecolor="white", edgecolor=gaya.INK2, linewidth=1.5,
               zorder=3, label="1445 H (2024)")
    ax.scatter(p25["hari_relatif"].reindex(yk), [ys[m] + 0.16 for m in yk],
               s=90, color=gaya.AKSEN, zorder=3, label="1446 H (2025)")
    ax.axvline(0.5, color=gaya.INK2, ls=":", lw=0.9)
    ax.text(0.7, len(yk) - 0.35, "hari H sampai sini", fontsize=8.5 * fs, color=gaya.INK2)
    ax.annotate("penyeberangan & udara 2024:\nmemuncak sebelum H", xy=(-3.3, 2.6),
                xytext=(-10.4, 1.1), fontsize=8.6 * fs, color=gaya.INK, ha="left",
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9))
    ax.set_yticks([ys[m] for m in yk])
    ax.set_yticklabels([m.replace("Angkutan ", "").replace("Perkeretaapian", "Kereta api")
                          .replace("SDP", "SDP (penyeberangan)") for m in yk],
                       fontsize=9 * fs)
    ax.set_xlim(-11, 13)
    ax.set_ylim(-0.8, len(yk) - 0.2)
    ax.set_xticks([-10, -7, -3, 0, 2, 5, 8, 12])
    ax.set_xticklabels([label_h(h) for h in [-10, -7, -3, 0, 2, 5, 8, 12]], fontsize=8.5 * fs)
    ax.legend(fontsize=8.6 * fs, loc="lower left", frameon=False)
    ax.grid(True, axis="x"); ax.grid(False, axis="y")
    gaya.simpan(fig, "02-puncak-per-moda", mode)


for mode in gaya.MODE:
    grafik_2(mode)

      event                    moda  hari_relatif  penumpang
ANGLEB 2024          Angkutan Jalan             4     322305
ANGLEB 2025          Angkutan Jalan             6     350831
ANGLEB 2024           Angkutan Laut             4     115310
ANGLEB 2025           Angkutan Laut             3     148272
ANGLEB 2024 Angkutan Perkeretaapian             4     242173
ANGLEB 2025 Angkutan Perkeretaapian             6     249888
ANGLEB 2024            Angkutan SDP            -3     389585
ANGLEB 2025            Angkutan SDP             6     391964
ANGLEB 2024          Angkutan Udara            -4     329507
ANGLEB 2025          Angkutan Udara             6     313981


## Temuan

> Bentuk puncak mudik Lebaran adalah dua bukit dengan lembah di hari H, dan bukit keduanya selalu lebih tinggi: bukit mudik menumpuk di H-3 (1,22 jt penumpang pada 1445 H, 1,22 jt pada 1446 H), lalu sepi saat Lebaran, dan bukit balik meledak H+4 sampai H+6 (1,35 jt dan 1,44 jt; puncak tertinggi kedua musim). Sebagian besar pergerakan terjadi SESUDAH hari Lebaran: 54-55% penumpang-hari jatuh sesudah dua hari Lebaran, dan hari median musimnya +2 (separuh musim baru berlalu setelah dua hari penuh pasca-Lebaran). Di 2024 penyeberangan dan pesawat masih memuncak sebelum H; di 2025 (cuti bersama diperpanjang) lima-modanya bergeser total: semua memuncak H+6. Dan rilis posko 1447 H menegaskan arah: realisasi selalu melampaui survei niat (143,9 juta direncanakan menjadi 147,55 juta pergerakan MPD).

In [5]:
def hari_median(ev):
    s = harian[ev].dropna().sort_index()
    kumul = s.cumsum() / s.sum()
    return int(kumul[kumul >= 0.5].index[0])

metrik = {
    "total_per_event": {ev: int(df[df.event == ev].penumpang.sum()) for ev in EV_ID},
    "puncak_harian": {ev: {"hari": int(harian[ev].idxmax()),
                           "penumpang": int(harian[ev].max())} for ev in EV_ID},
    "bukit_mudik_h3": {ev: int(harian.loc[-3, ev]) for ev in EV_ID[:2]},
    "kolam_hari_H": {ev: int(harian.loc[0, ev]) for ev in EV_ID[:2]},
    "share_sesudah_h2": {ev: round(float(harian.loc[2:, ev].sum() / harian[ev].sum()), 3)
                         for ev in EV_ID[:2]},
    "hari_median": {ev: hari_median(ev) for ev in EV_ID[:2]},
    "puncak_per_moda": {f"{r['event']}|{r['moda']}": int(r["hari_relatif"])
                        for r in puncak_moda.to_dict("records")},
    "kerangka_1447H": {"kumulatif_h8_sampai_h": 10_887_584,
                       "vs_1446H_persen": 8.58,
                       "proyeksi_vs_realisasi_mpd_juta": {"2025": ["146,48", "154,6"],
                                                        "2026": ["143,91", "147,55"]}},
}
print(json.dumps(metrik, indent=2, ensure_ascii=False))
Path("data/metrik-010.json").write_text(json.dumps(metrik, indent=2, ensure_ascii=False))

{
  "total_per_event": {
    "ANGLEB 2024": 21468508,
    "ANGLEB 2025": 23062930,
    "NATARU 2024 2025": 18319399
  },
  "puncak_harian": {
    "ANGLEB 2024": {
      "hari": 4,
      "penumpang": 1350087
    },
    "ANGLEB 2025": {
      "hari": 6,
      "penumpang": 1444983
    },
    "NATARU 2024 2025": {
      "hari": 5,
      "penumpang": 1014842
    }
  },
  "bukit_mudik_h3": {
    "ANGLEB 2024": 1249321,
    "ANGLEB 2025": 1221945
  },
  "kolam_hari_H": {
    "ANGLEB 2024": 681775,
    "ANGLEB 2025": 750203
  },
  "share_sesudah_h2": {
    "ANGLEB 2024": 0.503,
    "ANGLEB 2025": 0.5
  },
  "hari_median": {
    "ANGLEB 2024": 2,
    "ANGLEB 2025": 1
  },
  "puncak_per_moda": {
    "ANGLEB 2024|Angkutan Jalan": 4,
    "ANGLEB 2025|Angkutan Jalan": 6,
    "ANGLEB 2024|Angkutan Laut": 4,
    "ANGLEB 2025|Angkutan Laut": 3,
    "ANGLEB 2024|Angkutan Perkeretaapian": 4,
    "ANGLEB 2025|Angkutan Perkeretaapian": 6,
    "ANGLEB 2024|Angkutan SDP": -3,
    "ANGLEB 2025|Angkutan SDP":

1342

## Batas & cara reproduksi

Enam hal jujur. Pertama, ini penumpang **angkutan umum** yang dipantau posko; kendaraan pribadi, tulang punggung mudik Indonesia, di luar tabel, jadi bentuk ini membaca irama musim, bukan seluruh manusianya. Kedua, angka harian di luar jendela pantauan ditandai penerbit "belum direkonsiliasi" dan dipakai apa adanya. Ketiga, dua musim adalah titik, bukan tren kuat: pergeseran 2024 ke 2025 bersinggungan dengan kalender cuti bersama 1446 H, dan korelasi itu tidak bisa dipisahkan cuma dari dua musim. Keempat, "H" di katalog adalah hari Lebaran ganda pertama/kedua (H 1, H 2); bila tahun itu Lebaran resmi hanya satu hari menurut pengumuman, pemetaannya tetap mengikuti katalog. Kelima, Nataru turut dipajang di metrik hanya sebagai pembanding bayangan, bukan bahan klaim. Keenam, Angka 1447 H (2026) belum ada di katalog mesin-baca saat akses 30 September 2026, sehingga musim terbaru yang dianalisis adalah 1446 H.